# Phase 1 — Structured Streaming Fundamentals

**Concept.** Structured Streaming treats a live, growing data source as an
unbounded table, and re-expresses it as a series of small "micro-batch"
queries against whatever new data has arrived since the last run. You write
the *same* DataFrame transformations as batch Spark — the engine handles
incremental execution, fault tolerance (via checkpointing), and
exactly-once processing underneath.

Official docs: [Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)

**Why it matters professionally.** Real pipelines rarely get "all the data
at once" the way our Phase 1 medallion notebooks have so far (we always
had all 3 months' Parquet files sitting in `data/raw/` upfront). In
production, files/events arrive continuously — hourly trip extracts, Kafka
topics, CDC feeds — and reprocessing everything from scratch on every new
arrival doesn't scale. Structured Streaming is how Spark (and Databricks'
Auto Loader / Delta Live Tables / Lakeflow Pipelines) handle this: track
what's already been processed, and only touch what's new.

**Architecture decision.** Auto Loader (`cloudFiles` format) — the
Databricks-native incremental file-ingestion mechanism — is Databricks-only
and isn't available in local open-source Spark. Three realistic options for
teaching this:

| Approach | Description | Trade-off |
|---|---|---|
| A. Databricks-only | Skip local entirely, teach Auto Loader directly on the Databricks workspace | Loses the local-first validation loop we've used for everything else in Phase 1 |
| **B. Local file source, then Auto Loader by analogy (recommended)** | Teach core Structured Streaming concepts locally using Spark's built-in `parquet` file source against a simulated "landing" directory, then map the same API onto Auto Loader's `cloudFiles` format as a format-string swap | Small extra setup (simulating file arrival), but validates every concept locally first, consistent with how we did Delta Lake |
| C. Toy-only (`rate` source) | Only use the synthetic `rate` source, skip real file-based incremental ingestion | Doesn't touch checkpointing/incremental-offset semantics — the actual point of this topic |

**Recommendation: B.** Same local-first-then-port pattern we used for Delta
Lake — and Auto Loader really is just `spark.readStream.format("cloudFiles")`
instead of `spark.readStream.format("parquet")` on the same API, so nothing
learned here needs re-learning on Databricks.

**Where this fits in DataForge AI.**

```mermaid
flowchart LR
    A[Landing dir<br/>simulated file arrivals] -->|readStream.parquet| B[Streaming Bronze<br/>Delta table]
    B -->|readStream.format delta| C[Streaming Gold<br/>windowed aggregation]
```

This extends the same Bronze -> Silver -> Gold mental model from
`06_delta_lake_fundamentals.ipynb`, but built incrementally instead of in
one batch pass.


In [1]:
import os
import shutil
import time

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName("dataforge-streaming")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Driver memory:", spark.conf.get("spark.driver.memory"))

RAW = "../data/raw"
STREAM_ROOT = "../data/streaming"
LANDING_DIR = f"{STREAM_ROOT}/landing"
CHECKPOINT_BRONZE = f"{STREAM_ROOT}/checkpoints/bronze"
CHECKPOINT_GOLD = f"{STREAM_ROOT}/checkpoints/gold"
BRONZE_STREAM_PATH = f"{STREAM_ROOT}/bronze/trips"
GOLD_STREAM_PATH = f"{STREAM_ROOT}/gold/hourly_demand_streaming"

# Start clean each run -- streaming checkpoints are stateful, so a leftover
# checkpoint from a previous run would try to resume from wherever it left
# off instead of demonstrating incremental arrival from scratch.
shutil.rmtree(STREAM_ROOT, ignore_errors=True)
os.makedirs(LANDING_DIR, exist_ok=True)
print("Landing dir ready:", LANDING_DIR)


Driver memory: 4g
Landing dir ready: ../data/streaming/landing


## Toy example: the `rate` source

`rate` is a built-in synthetic source that emits `(timestamp, value)` rows
at a configurable rate (`rowsPerSecond`) — the simplest way to see the
micro-batch execution model without any real data. We write it to the
`memory` sink (a named in-process table queryable via `spark.sql`), let it
run for a few seconds, then stop it — this is the "hello world" of
Structured Streaming.


In [2]:
rate_stream = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

query = (
    rate_stream.writeStream
    .format("memory")
    .queryName("rate_demo")
    .outputMode("append")
    .trigger(processingTime="1 second")
    .start()
)

time.sleep(5)
query.stop()

print("Rows seen:", spark.sql("SELECT * FROM rate_demo").count())
spark.sql("SELECT * FROM rate_demo ORDER BY timestamp").show(10, truncate=False)


Rows seen: 20
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-09-09 15:18:59.642|0    |
|2026-09-09 15:18:59.842|1    |
|2026-09-09 15:19:00.042|2    |
|2026-09-09 15:19:00.242|3    |
|2026-09-09 15:19:00.442|4    |
|2026-09-09 15:19:00.642|5    |
|2026-09-09 15:19:00.842|6    |
|2026-09-09 15:19:01.042|7    |
|2026-09-09 15:19:01.242|8    |
|2026-09-09 15:19:01.442|9    |
+-----------------------+-----+
only showing top 10 rows



## Triggers

A trigger controls *when* the engine runs the next micro-batch. Doc:
[Triggers](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#triggers)

| Trigger | Behavior |
|---|---|
| Default (unspecified) | Runs the next micro-batch as soon as the previous one finishes |
| `processingTime="1 second"` | Fixed interval — used above for the `rate` demo |
| `availableNow=True` | Processes everything currently available in the source (splitting into as many micro-batches as needed), then stops on its own — the modern, safer replacement for the now-deprecated `once=True` |
| `continuous="1 second"` | Experimental low-latency (~1ms) record-at-a-time mode; far fewer operations supported, rarely used in practice |

We'll use `availableNow=True` below — it's the trigger that best fits a
Databricks Job task (run, process what's new, stop, exit cleanly) rather
than a long-running always-on stream.


## Simulating incremental file arrival

Auto Loader (`cloudFiles` format) is Databricks-only, so we simulate the
same core idea locally: a "landing" directory that starts nearly empty and
gets new files copied into it over time, standing in for files arriving
from an upstream system. We read it with Spark's built-in `parquet`
streaming file source and a checkpoint location — the checkpoint is what
tracks *which files have already been processed*, so re-running the same
query later only picks up genuinely new files, never reprocessing old ones.

This is exactly the mechanism Auto Loader scales up for cloud storage
(via file-notification services instead of directory listing) — same
`readStream` API, same checkpoint-based incremental-offset idea.


In [3]:
# Structured Streaming's file source needs an explicit schema up front --
# unlike batch reads, it won't re-inspect the whole (growing) directory on
# every micro-batch for performance/consistency reasons. Parquet embeds its
# own schema, so we borrow it from a static batch read of one file.
static_schema = spark.read.parquet(f"{RAW}/yellow_tripdata_2023-01.parquet").schema

# Simulate the first month "arriving" -- copy just one file into the landing dir.
shutil.copy(
    f"{RAW}/yellow_tripdata_2023-01.parquet",
    f"{LANDING_DIR}/yellow_tripdata_2023-01.parquet",
)
print("Landing dir:", os.listdir(LANDING_DIR))


Landing dir: ['yellow_tripdata_2023-01.parquet']


## Streaming Bronze: incremental ingest with `trigger(availableNow=True)`

Same column-casting logic as batch (`TARGET_TYPES` from `dataforge_ai`,
imported unchanged), but the *reading* is `readStream` instead of `read`,
and the *writing* has a `checkpointLocation`. Note: `read_and_cast`/
`read_trips` themselves aren't reusable here — they call `spark.read` on a
single static path, which is inherently batch-only. Only the shared
`TARGET_TYPES` dict ports over unchanged.


In [4]:
from dataforge_ai.io import TARGET_TYPES
from pyspark.sql import functions as F


def cast_stream(df):
    df = df.toDF(*[c.lower() for c in df.columns])
    for col, target in TARGET_TYPES.items():
        df = df.withColumn(col, F.col(col.lower()).cast(target))
    return df.select(*TARGET_TYPES.keys())


raw_stream = spark.readStream.schema(static_schema).parquet(LANDING_DIR)
raw_stream = cast_stream(raw_stream)

bronze_query = (
    raw_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_BRONZE)
    .trigger(availableNow=True)
    .start(BRONZE_STREAM_PATH)
)
bronze_query.awaitTermination()

bronze_stream_table = spark.read.format("delta").load(BRONZE_STREAM_PATH)
print("Bronze rows after 1st arrival:", bronze_stream_table.count())


Bronze rows after 1st arrival: 3066766


## A second file arrives

Copy in the February file and re-run the *exact same* streaming query. If
incremental ingestion is working, Bronze's row count should grow by
roughly one month's worth of trips — not double, and not re-count January.


In [5]:
shutil.copy(
    f"{RAW}/yellow_tripdata_2023-02.parquet",
    f"{LANDING_DIR}/yellow_tripdata_2023-02.parquet",
)
print("Landing dir:", os.listdir(LANDING_DIR))

raw_stream = spark.readStream.schema(static_schema).parquet(LANDING_DIR)
raw_stream = cast_stream(raw_stream)

bronze_query = (
    raw_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_BRONZE)
    .trigger(availableNow=True)
    .start(BRONZE_STREAM_PATH)
)
bronze_query.awaitTermination()

bronze_stream_table = spark.read.format("delta").load(BRONZE_STREAM_PATH)
print("Bronze rows after 2nd arrival:", bronze_stream_table.count())
(
    bronze_stream_table
    .groupBy(F.date_format("tpep_pickup_datetime", "yyyy-MM").alias("pickup_month"))
    .count()
    .orderBy("pickup_month")
    .show()
)


Landing dir: ['yellow_tripdata_2023-01.parquet', 'yellow_tripdata_2023-02.parquet']


StreamingQueryException: [STREAM_FAILED] Query [id = 7d84b4d2-c923-4a2d-96ac-9c2288aa4337, runId = ab481fd1-64a5-43a9-9243-38de2f6421a1] terminated with exception: Job aborted due to stage failure: Task 5 in stage 21.0 failed 1 times, most recent failure: Lost task 5.0 in stage 21.0 (TID 244) (e0139c99f795 executor driver): org.apache.spark.SparkException: Parquet column cannot be converted in file file:///home/jovyan/work/data/streaming/landing/yellow_tripdata_2023-02.parquet. Column: [VendorID], Expected: bigint, Found: INT32.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.unsupportedSchemaColumnConvertError(QueryExecutionErrors.scala:855)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:287)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:593)
	at org.apache.spark.sql.catalyst.expressi

## Streaming Gold: windowed aggregation + watermarking

`hourly_demand()` from `dataforge_ai.aggregations` uses arbitrary
`Window.partitionBy(...).orderBy(...)` for ranking and running totals —
Structured Streaming doesn't support arbitrary row-level windows like that,
since it would require unbounded state (the engine can never know it's
"seen everything" for an arbitrary order). Streaming aggregation instead
uses **time-based `window()` grouping**, bounded by a **watermark** that
tells the engine how long to wait for late-arriving data before finalizing
a window and discarding its state. Doc:
[Handling late data and watermarking](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#handling-late-data-and-watermarking)

So `hourly_demand()` doesn't port directly either — this is a second real
example (alongside `read_trips`) of a batch function whose *logic* doesn't
translate 1:1 to streaming, even though the underlying data and intent are
the same.


In [6]:
bronze_stream_source = spark.readStream.format("delta").load(BRONZE_STREAM_PATH)

gold_stream = (
    bronze_stream_source
    .withWatermark("tpep_pickup_datetime", "1 hour")
    .groupBy(
        F.window("tpep_pickup_datetime", "1 hour").alias("pickup_window"),
        "PULocationID",
    )
    .agg(F.count("*").alias("trip_count"))
)

gold_query = (
    gold_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_GOLD)
    .outputMode("append")
    .trigger(availableNow=True)
    .start(GOLD_STREAM_PATH)
)
gold_query.awaitTermination()

gold_stream_table = spark.read.format("delta").load(GOLD_STREAM_PATH)
print("Streaming Gold rows:", gold_stream_table.count())
gold_stream_table.orderBy("pickup_window").show(10, truncate=False)


Streaming Gold rows: 71440
+------------------------------------------+------------+----------+
|pickup_window                             |PULocationID|trip_count|
+------------------------------------------+------------+----------+
|{2008-12-31 23:00:00, 2009-01-01 00:00:00}|132         |1         |
|{2008-12-31 23:00:00, 2009-01-01 00:00:00}|7           |1         |
|{2022-10-24 17:00:00, 2022-10-24 18:00:00}|1           |1         |
|{2022-10-24 20:00:00, 2022-10-24 21:00:00}|17          |1         |
|{2022-10-24 21:00:00, 2022-10-24 22:00:00}|48          |1         |
|{2022-10-24 23:00:00, 2022-10-25 00:00:00}|211         |1         |
|{2022-10-25 00:00:00, 2022-10-25 01:00:00}|265         |1         |
|{2022-10-25 00:00:00, 2022-10-25 01:00:00}|132         |1         |
|{2022-10-25 03:00:00, 2022-10-25 04:00:00}|1           |1         |
|{2022-10-25 07:00:00, 2022-10-25 08:00:00}|132         |1         |
+------------------------------------------+------------+----------+
only sh

## Recap

- Structured Streaming reuses the *same* DataFrame transformation API as
  batch — only reading (`readStream`), writing (`checkpointLocation`), and
  aggregation semantics (time-`window()` + watermark vs. arbitrary `Window`)
  differ.
- Package reuse boundary: the shared `TARGET_TYPES` dict ported unchanged;
  `read_and_cast`/`read_trips` (batch-only, call `spark.read` on a static
  path) and `hourly_demand` (arbitrary `Window.partitionBy/orderBy` ranking)
  did **not** port directly — both needed a streaming-appropriate rewrite.
- Learned: the `rate` source, output modes, triggers (`processingTime`,
  `availableNow`), checkpoint-based incremental/exactly-once ingestion, and
  watermarking for bounded late-data handling.
- Databricks Auto Loader (`cloudFiles` format) is the production-scale
  version of exactly what we built with
  `spark.readStream.schema(...).parquet(LANDING_DIR)` — same API, swapped
  format string, adds cloud-native file notification (instead of directory
  listing) and automatic schema inference/evolution tracked in a schema
  location. Doc:
  [Auto Loader](https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/)

## Challenge

1. Copy the March file into the landing dir and re-run the Bronze cell's
   pattern — confirm the row count grows a third time, incrementally.
2. Change the Gold query's `outputMode("append")` to `outputMode("complete")`
   and re-run — observe the error. `append` mode needs a watermark to know
   when a window is "final" enough to emit; `complete` mode re-emits the
   *entire* result table every trigger regardless of watermark, which is
   why it's used for continuously-updated dashboards, not incremental
   append-only sinks like ours.
